In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

silver_df = spark.read.table('fraud_mlops.silver.transactions_clean_incremental')

gold_base_df = silver_df.withColumn(
    'transaction_id',
    sha2(
        concat_ws(
            '||',
            col('step').cast('string'),
            col('transaction_type'),
            col('amount').cast('string'),
            col('origin_account'),
            col('destination_account')
        ),256
    )    
)

gold_base_df = (
    gold_base_df

    .withColumn(
        "hour_of_day",
        ((col("step") - 1) % 24).cast("integer")
    )

    .withColumn(
        "day_number",
        (floor((col("step") - 1) / 24) + 1).cast("integer")
    )
)

gold_base_df = gold_base_df.withColumn('log_amount',log1p(col('amount')))

gold_base_df = gold_base_df.withColumn('type_cash_in',when(col('transaction_type')=='CASH_IN',1).otherwise(0))\
    .withColumn('type_cash_out',when(col('transaction_type')=='CASH_OUT',1).otherwise(0))\
        .withColumn('type_debit',when(col('transaction_type')=='DEBIT',1).otherwise(0))\
            .withColumn('type_payment',when(col('transaction_type')=='PAYMENT',1).otherwise(0))\
                .withColumn('type_transfer',when(col('transaction_type')=='TRANSFER',1).otherwise(0))


origin_history_window = Window.partitionBy(col('origin_account')).orderBy(col('step')).rangeBetween(Window.unboundedPreceding, -1)


gold_base_df = gold_base_df.withColumn('origin_txn_count_prior',count('*').over(origin_history_window))\
    .withColumn('origin_avg_amount_prior',avg(col('amount')).over(origin_history_window))\
        .withColumn('origin_total_amount_prior',sum(col('amount')).over(origin_history_window))


gold_base_df = gold_base_df.withColumn('amount_vs_origin_avg',when(col('origin_avg_amount_prior')>0,col('amount')/col('origin_avg_amount_prior')).otherwise(0.0))

gold_base_df = gold_base_df.withColumn('origin_has_history',when(col('origin_txn_count_prior')>0,1).otherwise(0))\
    .fillna({
        'origin_txn_count_prior':0,
        'origin_avg_amount_prior':0.0,
        'origin_total_amount_prior':0.0,
        'amount_vs_origin_avg':0.0
    })

gold_df = gold_base_df.select('transaction_id',\
    'step','hour_of_day','day_number',\
    'amount','log_amount',\
    'type_cash_in','type_cash_out','type_debit','type_payment','type_transfer',\
    'origin_txn_count_prior','origin_avg_amount_prior','origin_total_amount_prior','amount_vs_origin_avg','origin_has_history',\
    'is_fraud','label_status')

gold_df=gold_df.withColumn('dataset_split',when(col('step')<=520,'train')\
    .when(col('step')<=632,'validation')\
        .otherwise('test'))

gold_df.write.format('delta').mode('overwrite').option('overwriteSchema',True).saveAsTable('fraud_mlops.gold.transaction_features_incremental')


training_snapshot_df =gold_df.filter(col('label_status')=='AVAILABLE')\
    .drop(col('label_status'))


training_snapshot_df.write.format('delta').mode('overwrite').option('overwriteSchema',True).saveAsTable('fraud_mlops.gold.fraud_training_snapshot_incremental')


print("Silver -> Gold incremental processing completed.")



In [0]:
%sql

SELECT
    dataset_split,
    COUNT(*) AS rows,
    SUM(is_fraud) AS fraud_rows
FROM fraud_mlops.gold.fraud_training_snapshot_incremental
GROUP BY dataset_split
ORDER BY dataset_split;

In [0]:
%sql

SELECT
    'HISTORICAL_TRAINING' AS source,
    COUNT(*) AS rows,
    AVG(amount) AS avg_amount,
    AVG(log_amount) AS avg_log_amount,
    AVG(type_cash_in) AS cash_in_rate,
    AVG(type_cash_out) AS cash_out_rate,
    AVG(type_debit) AS debit_rate,
    AVG(type_payment) AS payment_rate,
    AVG(type_transfer) AS transfer_rate,
    AVG(origin_txn_count_prior) AS avg_origin_txn_count_prior,
    AVG(origin_has_history) AS origin_history_rate,
    AVG(is_fraud) AS fraud_rate
FROM fraud_mlops.gold.transaction_features

UNION ALL

SELECT
    'PRODUCTION_40K' AS source,
    COUNT(*) AS rows,
    AVG(amount) AS avg_amount,
    AVG(log_amount) AS avg_log_amount,
    AVG(type_cash_in) AS cash_in_rate,
    AVG(type_cash_out) AS cash_out_rate,
    AVG(type_debit) AS debit_rate,
    AVG(type_payment) AS payment_rate,
    AVG(type_transfer) AS transfer_rate,
    AVG(origin_txn_count_prior) AS avg_origin_txn_count_prior,
    AVG(origin_has_history) AS origin_history_rate,
    AVG(is_fraud) AS fraud_rate
FROM fraud_mlops.gold.transaction_features_incremental;

In [0]:
%sql

CREATE OR REPLACE VIEW fraud_mlops.gold.production_normal_drift_reference
AS

SELECT
    amount,
    log_amount,
    type_cash_in,
    type_cash_out,
    type_debit,
    type_payment,
    type_transfer,
    origin_txn_count_prior,
    origin_avg_amount_prior,
    origin_total_amount_prior,
    amount_vs_origin_avg,
    origin_has_history

FROM fraud_mlops.gold.production_normal_reference;

In [0]:
%sql

DESCRIBE fraud_mlops.gold.production_normal_drift_reference;

In [0]:
%sql
CREATE OR REPLACE TABLE
fraud_mlops.gold.production_drift_window
AS

SELECT g.*
FROM fraud_mlops.gold.transaction_features_incremental g

LEFT ANTI JOIN
fraud_mlops.gold.production_normal_reference n

ON g.transaction_id = n.transaction_id;

In [0]:
%sql
SELECT COUNT(*)
FROM fraud_mlops.gold.production_drift_window;

In [0]:
%sql
SELECT COUNT(*)
FROM fraud_mlops.gold.transaction_features_incremental;

In [0]:
%sql
CREATE OR REPLACE VIEW
fraud_mlops.gold.production_drift_reference
AS

SELECT
    amount,
    log_amount,
    type_cash_in,
    type_cash_out,
    type_debit,
    type_payment,
    type_transfer,
    origin_txn_count_prior,
    origin_avg_amount_prior,
    origin_total_amount_prior,
    amount_vs_origin_avg,
    origin_has_history
FROM fraud_mlops.gold.production_drift_window;